# RuleTaker and FOLIO Datasets Demo

## Overview
This notebook demonstrates a pipeline that loads two high-quality neuro-symbolic reasoning benchmarks from HuggingFace and standardizes them to a unified schema for logical reasoning evaluation.

**Dataset 1: RuleTaker** (165,833 examples)
- Natural language context (facts + rules) with entailment/not-entailment labels
- Supports proof-failure classification and type-based LLM dispatch
- Configurable reasoning depth (0-5 hops)

**Dataset 2: FOLIO** (1,001 examples)
- Expert-written premises and conclusions with True/False/Uncertain labels
- Parallel first-order logic (FOL) annotations for each example
- Enables direct neuro-symbolic grounding

Together, these datasets provide scale (RuleTaker) and formal grounding depth (FOLIO) for neuro-symbolic reasoning pipelines.

## Setup: Install Dependencies

The cell below installs required packages. It uses a conditional pattern that ensures compatibility with both Google Colab and local Jupyter environments:
- On Colab: skips pre-installed core packages (numpy, pandas, etc.)
- Locally: installs packages at versions that match Colab's environment

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Non-Colab packages (always install)
_pip('loguru==0.7.2')

# Core packages (pre-installed on Colab, install locally to match Colab environment)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2')

## Imports

Import required modules for data loading, JSON processing, and visualization.

In [ ]:
import json
import sys
from pathlib import Path
from loguru import logger
import pandas as pd
import matplotlib.pyplot as plt

## Configuration

Set tunable parameters for the demo. Start with minimal values to ensure the notebook completes quickly.

In [ ]:
# Data loading parameters
MAX_EXAMPLES_PER_DATASET = 3  # Minimal: start with 3, scale up as needed
VERBOSE_LOGGING = True

# Output parameters
SHOW_SAMPLE_INPUTS = True
SHOW_METADATA = True

## Data Loading

Define a helper function to load the mini dataset from GitHub (with local fallback for offline environments).

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-6d242d-failure-mode-typed-neural-abduction-stru/main/round-1/dataset-1/demo/mini_demo_data.json"

def load_data():
    """Load mini demo dataset from GitHub URL or local file."""
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    
    if Path("mini_demo_data.json").exists():
        with open("mini_demo_data.json") as f:
            return json.load(f)
    
    raise FileNotFoundError("Could not load mini_demo_data.json from GitHub or local path")

## Load Data

Execute the data loading function.

In [ ]:
data = load_data()
print(f"Data loaded successfully.")
print(f"Datasets: {data['metadata']['datasets']}")

## Setup Logging

Configure logging for the data processing pipeline.

In [ ]:
if VERBOSE_LOGGING:
    logger.remove()
    logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")
else:
    logger.remove()
    logger.add(sys.stdout, level="WARNING", format="{level:<7}|{message}")

## Process and Standardize Data

Extract and standardize examples from both datasets. The examples are already in the unified schema from the source artifact, so we filter and display them here.

In [ ]:
# Store processed examples by dataset
processed_datasets = {}

for dataset_info in data['datasets']:
    dataset_name = dataset_info['dataset']
    examples = dataset_info['examples']
    
    # Limit to MAX_EXAMPLES_PER_DATASET for demo purposes
    limited_examples = examples[:MAX_EXAMPLES_PER_DATASET]
    
    logger.info(f"{dataset_name.upper()}: {len(limited_examples)} examples (from {len(examples)} total)")
    
    processed_datasets[dataset_name] = {
        'examples': limited_examples,
        'total_count': len(examples),
        'demo_count': len(limited_examples)
    }

## Analyze Dataset Structure

Examine the schema and metadata of each dataset.

In [ ]:
# Analyze RuleTaker structure
if 'ruletaker' in processed_datasets and processed_datasets['ruletaker']['examples']:
    rt_example = processed_datasets['ruletaker']['examples'][0]
    logger.info("RuleTaker example fields: " + str(list(rt_example.keys())))
    
    if SHOW_METADATA:
        logger.info(f"  - split: {rt_example.get('metadata_split')}")
        logger.info(f"  - config: {rt_example.get('metadata_config')}")
        logger.info(f"  - answer_type: {rt_example.get('metadata_answer_type')}")
        logger.info(f"  - output: {rt_example.get('output')}")

# Analyze FOLIO structure
if 'folio' in processed_datasets and processed_datasets['folio']['examples']:
    folio_example = processed_datasets['folio']['examples'][0]
    logger.info("FOLIO example fields: " + str(list(folio_example.keys())))
    
    if SHOW_METADATA:
        logger.info(f"  - split: {folio_example.get('metadata_split')}")
        logger.info(f"  - task_type: {folio_example.get('metadata_task_type')}")
        logger.info(f"  - answer_type: {folio_example.get('metadata_answer_type')}")
        logger.info(f"  - output: {folio_example.get('output')}")

## Sample Inputs

Display sample input/output pairs from each dataset.

In [ ]:
if SHOW_SAMPLE_INPUTS:
    # RuleTaker sample
    if 'ruletaker' in processed_datasets and processed_datasets['ruletaker']['examples']:
        print("\n" + "="*80)
        print("RULETAKER SAMPLE")
        print("="*80)
        ex = processed_datasets['ruletaker']['examples'][0]
        print(f"Input (truncated to 300 chars):\n{ex['input'][:300]}...\n")
        print(f"Output: {ex['output']}")
        print(f"Config: {ex.get('metadata_config')}")
    
    # FOLIO sample
    if 'folio' in processed_datasets and processed_datasets['folio']['examples']:
        print("\n" + "="*80)
        print("FOLIO SAMPLE")
        print("="*80)
        ex = processed_datasets['folio']['examples'][0]
        print(f"Input (truncated to 300 chars):\n{ex['input'][:300]}...\n")
        print(f"Output: {ex['output']}")
        print(f"Story ID: {ex.get('metadata_story_id')}")
        print(f"Example ID: {ex.get('metadata_example_id')}")

## Summary and Statistics

Display key statistics about the loaded datasets.

In [ ]:
# Create summary statistics
summary_data = []

for dataset_name, info in processed_datasets.items():
    summary_data.append({
        'Dataset': dataset_name.upper(),
        'Demo Examples': info['demo_count'],
        'Total Examples': info['total_count'],
        'Task Type': info['examples'][0].get('metadata_task_type', 'N/A') if info['examples'] else 'N/A',
        'Answer Type': info['examples'][0].get('metadata_answer_type', 'N/A') if info['examples'] else 'N/A'
    })

summary_df = pd.DataFrame(summary_data)
print("\n" + "="*80)
print("DATASET SUMMARY")
print("="*80)
print(summary_df.to_string(index=False))
print(f"\nTotal demo examples loaded: {sum(d['Demo Examples'] for d in summary_data)}")
print(f"Total examples in full datasets: {sum(d['Total Examples'] for d in summary_data)}")

## Visualization

Create visualizations of dataset composition and characteristics.

In [ ]:
# Visualization: Dataset sizes comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Total examples per dataset
dataset_names = [d['Dataset'] for d in summary_data]
total_counts = [d['Total Examples'] for d in summary_data]
axes[0].bar(dataset_names, total_counts, color=['#3498db', '#e74c3c'])
axes[0].set_ylabel('Number of Examples')
axes[0].set_title('Total Examples per Dataset')
axes[0].set_yscale('log')
for i, v in enumerate(total_counts):
    axes[0].text(i, v, str(v), ha='center', va='bottom')

# Demo vs Total comparison
demo_counts = [d['Demo Examples'] for d in summary_data]
x = range(len(dataset_names))
width = 0.35
axes[1].bar([i - width/2 for i in x], demo_counts, width, label='Demo', color='#2ecc71')
axes[1].bar([i + width/2 for i in x], total_counts, width, label='Total', color='#95a5a6')
axes[1].set_ylabel('Number of Examples')
axes[1].set_title('Demo vs Total Examples')
axes[1].set_xticks(x)
axes[1].set_xticklabels(dataset_names)
axes[1].legend()
axes[1].set_yscale('log')

plt.tight_layout()
plt.show()

print("\nVisualization complete. Datasets are ready for neuro-symbolic reasoning evaluation.")